# UNDERTONE - smoke test

Run this **before** any sweep. It costs ~15 minutes of the weekly 30 GPU-hour
quota and it is the difference between finding a broken adapter now and finding
it in six hours of zeros.

Each adapter is checked for:

| check | failure it catches |
|---|---|
| `loads` | gated repo, wrong auto-class, OOM across 2xT4 |
| `finite` | bf16-trained weights overflowing in fp16 on sm75 -- a NaN row argmaxes to "A" |
| `discriminates` | all four letter logits equal: wrong token ids, or a prompt the model never sees as a question |
| `parses` | free generation that never emits a bare letter |
| `deterministic` | sampling left on. The retired suite's "same" config gave 0.14 and 0.28 on two runs |
| `truncates_loudly` | over-long audio setting the flag instead of vanishing silently |

Models are loaded one at a time and unloaded immediately, so peak VRAM is one
model's worth, not thirteen.


In [ ]:
# Pinned for this model. If `load()` fails, this cell is the first thing to change.
%pip install -q "transformers>=4.57.1"
%pip install -q "accelerate>=1.0.0"
%pip install -q "librosa>=0.10.2"
%pip install -q "soundfile>=0.12.1"
print("--- resolved versions (freeze these before the paper run) ---")
import importlib.metadata as md
for pkg in ["transformers", "accelerate", "torch", "librosa"]:
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} not installed")

In [ ]:
import os, random, sys, json
import numpy as np, torch

SEED = 20260904
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Weights go to /kaggle/temp: scratch, and it does NOT count against the 20 GB
# /kaggle/working output cap. A 16-18 GB checkpoint in /kaggle/working would
# fail the commit at the end of the session.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Gated model. The token is resolved after the repo is cloned (next cell), from:
#   HF_TOKEN in the environment -> Kaggle secret named HF_TOKEN -> .hf_token at
#   the repo root (gitignored - the repo is public, so a token in tracked source
#   would be scraped from GitHub within minutes).
# Accept the licence on the Hub with the account that owns the token first.
GATED = True

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB  sm{p.major}{p.minor}")
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    print("\nsm < 80: no bf16 compute and no flash-attention-2. "
          "Every adapter loads in fp16 for this reason.")

In [ ]:
REPO_URL = "https://github.com/DeepanIsCool/longaudiobench.git"
REPO_REF = "undertone"   # pin to a commit sha before the paper run

import subprocess, shutil, os, sys
if os.path.exists("/kaggle/working/longaudiobench"):
    shutil.rmtree("/kaggle/working/longaudiobench")
for attempt in range(3):
    rc = subprocess.call(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                          REPO_URL, "/kaggle/working/longaudiobench"])
    if rc == 0:
        break
else:
    raise RuntimeError("could not clone the benchmark repo")

sys.path.insert(0, "/kaggle/working/longaudiobench")
import importlib; importlib.invalidate_caches()

from undertone import ItemPack, adapters, env, runner, scoring
print("adapters registered:", len(adapters.list_adapters()))

if env.export_hf_token():
    print("HF token resolved")
elif globals().get("GATED"):
    raise RuntimeError(
        "this model is gated and no token was found. Add a Kaggle secret named "
        "HF_TOKEN, or write the token to .hf_token at the repo root.")

hw = env.resolve_hardware()
print(f"hardware: {hw.detail}  dtype={hw.dtype}  signature={hw.signature}")
print(f"versions: {env.versions()}")
# Every result row is stamped with this signature. The analysis refuses to put
# two signatures in one table -- a benchmark whose rows came from different
# backends compares machines, not models.

In [ ]:
from undertone.adapters.base import get_adapter
for key in adapters.list_adapters():
    a = get_adapter(key)
    print(f"{key:26s} {a.max_audio_s:7.0f}s  {a.primary:8s} {a.model_id}")

In [ ]:
# One at a time: 13 models will not co-reside on 32 GB.
# Start with the two that bracket the roster -- Aero (~4 GB, 15 min ceiling) and
# Qwen2-Audio (16.8 GB, 30 s ceiling) -- then widen once both are green.
# Smallest first, so a disk or driver problem surfaces in minutes rather than
# after a 17 GB download. Gated models last: they fail fast without a token and
# that failure should not sit in front of twelve models that would have passed.
KEYS = [
    "aero_1_audio",            # ~4 GB, brackets the light end
    "voxtral_mini_3b",         # 9.5 GB
    "moss_audio_4b_instruct",  # 10.4 GB, bf16 mel override
    "moss_audio_4b_thinking",  # free-gen primary
    "phi4_multimodal",         # 11 GB, own transformers pin
    "qwen2_5_omni_3b",         # 11 GB, disable_talker
    "audio_flamingo_next",     # 16.5 GB, 2xT4
    "qwen2_audio_7b",          # 16.8 GB, 30 s cap - brackets the heavy end
    "moss_audio_8b_instruct",  # 18 GB, 2xT4
    "moss_audio_8b_thinking",
    "qwen2_5_omni_7b",         # 21 GB
    "gemma3n_e2b",             # gated
    "gemma3n_e4b",             # gated
]

from undertone.smoke import disk_free_gb, purge_cache, smoke_adapter

# ~130 GB of checkpoints against ~60 GB of Kaggle disk: each model's weights are
# deleted after it is checked, or the run dies on a download partway through
# rather than on anything worth knowing.
PURGE_AFTER_EACH = True

reports = []
for key in KEYS:
    print(f"\n{'=' * 72}\n{key}   (disk free: {disk_free_gb()} GB)\n{'=' * 72}")
    adapter = get_adapter(key)
    report = smoke_adapter(adapter)
    reports.append(report)
    for name, result in report["checks"].items():
        print(f"  {'PASS' if result['ok'] else 'FAIL'}  {name}: {result['detail']}")
    if report.get("traceback"):
        print(report["traceback"])
    if PURGE_AFTER_EACH:
        print(f"  freed {purge_cache(adapter.model_id)} GB")

with open("/kaggle/working/smoke_report.json", "w") as fh:
    json.dump(reports, fh, indent=2, default=str)

In [ ]:
bad = [r for r in reports if not r.get("ok")]
for r in reports:
    print(f"{'ok  ' if r.get('ok') else 'FAIL'} {r['key']:26s} "
          f"{r.get('peak_vram_gb', '?')} GB  {r.get('seconds', '?')}s  "
          f"{'failed: ' + ', '.join(r['failures']) if r['failures'] else ''}")

assert not bad, f"fix these adapters before spending quota on a sweep: {[r['key'] for r in bad]}"
print("\nall green - safe to run a sweep")

In [ ]:
# Optional and expensive. MOSS-Audio's ceiling is inferred from config.json
# (12.5 tokens/s against a 40 960 context) and is not documented by its authors,
# so this is how the truncation table gets a measured number.
from undertone.smoke import measure_ceiling

MEASURE = []          # e.g. ["moss_audio_4b_instruct"]
for key in MEASURE:
    print(json.dumps(measure_ceiling(get_adapter(key)), indent=2))